# 1. Import Libraries

In [1]:
import torch as t
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn import preprocessing
import pytorch_lightning as pl
from bmi.benchmark.tasks import task_multinormal_sparse, transform_spiral_task, transform_half_cube_task
from MIND_Unet_static_HM_MOE import MINDEstimator, logistic_integrate

plt.style.use('seaborn-v0_8-paper')

ModuleNotFoundError: No module named 'torch'

# 2. Utilities

In [127]:
def calculate_mi_trapz(conditional, unconditional, logsnr):
    """Calculate MI using trapezoidal integration."""
    return (t.trapz(unconditional, logsnr) - t.trapz(conditional, logsnr)) * 0.5


def calculate_orthogonal_mse(unconditional_ehat, conditional_ehat, num_points, test_sample_num):
    """Calculate MSE based on orthogonal principle."""
    unconditional_ehat = unconditional_ehat.view(num_points * test_sample_num, -1)
    conditional_ehat = conditional_ehat.view(num_points * test_sample_num, -1)
    error = (unconditional_ehat - conditional_ehat).flatten(start_dim=1)
    error = t.einsum('ij,ij->i', error, error)
    error = error.view(num_points, test_sample_num)
    return error.mean(dim=1)

# 3. Load Dataset

In [91]:
def process_data(task, train_sample_num=100000, test_sample_num=10000):
    """Process data of different tasks for model evaluation"""
    # Prepare data
    pl.seed_everything(0)
    X, Y = task.sample(train_sample_num + test_sample_num, seed=0)
    X, Y = X.__array__(), Y.__array__()
    
    scaler_X = preprocessing.StandardScaler()
    scaler_Y = preprocessing.StandardScaler()
    scaler_X.fit(X[:train_sample_num])
    scaler_Y.fit(Y[:train_sample_num])
    
    X = scaler_X.transform(X)
    Y = scaler_Y.transform(Y)
    X_test = X[train_sample_num:]
    Y_test = Y[train_sample_num:]
    
    return X_test, Y_test

strength = 2000 # Strength of correlation
dim = 3 # Dimensionality of data

# multinormal sparse dataset
task = task_multinormal_sparse(dim_x=dim, dim_y=dim, strength=strength)
task.task_name = f"multinormal_sparse_{strength}_dim_{dim}"

# spiral multinormal sparse dataset
spiral_cube_task = transform_spiral_task(task)            
spiral_cube_task.task_name = f"spiral_multinormal_sparse_{strength}_dim_{dim}"

# half cube multinormal sparse dataset
half_cube_task = transform_half_cube_task(task)
half_cube_task.task_name = f"half_cube_multinormal_sparse_{strength}_dim_{dim}"

# ['multinormal_sparse_2000_dim_3', 'spiral_multinormal_sparse_2000_dim_3', 'half_cube_multinormal_sparse_2000_dim_3']
X_test, Y_test = process_data(task=task, train_sample_num=100000, test_sample_num=10000) # Used for the following experiments
X_spiral_test, Y_spiral_test = process_data(task=spiral_cube_task, train_sample_num=100000, test_sample_num=10000)
X_half_cude_test, Y_half_cude_test = process_data(task=half_cube_task, train_sample_num=100000, test_sample_num=10000)

task_info = {
        'name': task.task_name,
        'dim': dim,
        'strength': strength,
        'mi': task.mutual_information
}

Seed set to 0


Seed set to 0
Seed set to 0


# 4. Calculate Theoretical MMSE curves

In [149]:
def setup_model_evaluation(num_points=100):
    """Setup model evaluation parameters"""
    static_location = 5
    static_scale = 4
    logsnr, weights = logistic_integrate(static_location, static_scale, num_points)
    logsnr, sort_indices = t.sort(logsnr)
    weights = weights[sort_indices]
    
    return logsnr, weights

logsnr, weights = setup_model_evaluation(num_points=100)
mmse_curves = {}

In [150]:
def compute_theoretical_mmse(X, Y, logsnr):
    """
    Compute MMSE using the theoretical formula.
    """
    # add noise
    alpha = 1 / (1 + np.exp(-logsnr))  # sigmoid(logsnr)
    beta = 1 / (1 + np.exp(logsnr))   # sigmoid(-logsnr)
    epsilon = np.random.randn(*X.shape)
    Z = np.sqrt(alpha) * X + np.sqrt(beta) * epsilon
    
    # Get dimensions
    dim_x = X.shape[1]
    
    # Compute base covariance matrix for X
    Sigma_xx = np.cov(X.T)
    
    # First compute unconditional MMSE (X|Z)
    Sigma_xz = np.cov(X.T, Z.T)[:dim_x, dim_x:]
    Sigma_zz = np.cov(Z.T)
    mmse_uncond = np.trace(Sigma_xx - Sigma_xz @ np.linalg.inv(Sigma_zz) @ Sigma_xz.T)
    
    if Y is not None and Y.shape[1] > 0:
        # Concatenate Z and Y to form joint observation [Z,Y]
        ZY = np.hstack([Z, Y])
        
        # Compute joint covariance matrices
        Sigma_x_zy = np.cov(X.T, ZY.T)[:dim_x, dim_x:]
        Sigma_zy = np.cov(ZY.T)
        
        # Compute conditional covariance
        Sigma_x_given_zy = Sigma_xx - Sigma_x_zy @ np.linalg.inv(Sigma_zy) @ Sigma_x_zy.T
        mmse_cond = np.trace(Sigma_x_given_zy)
    else:
        mmse_cond = mmse_uncond
    
    return mmse_cond, mmse_uncond

def generate_theoretical_curves(logsnrs, X, Y, num_points=100):
    """Generate analytic MMSE curves for multinormal distribution"""
    mmse_cond = np.zeros(num_points)
    mmse_uncond = np.zeros(num_points)
    
    for i, logsnr in enumerate(logsnrs):
        mmse_c, mmse_u = compute_theoretical_mmse(X, Y, logsnr)
        mmse_cond[i] = mmse_c
        mmse_uncond[i] = mmse_u
    mmse_cond *= np.exp(logsnrs)
    mmse_uncond *= np.exp(logsnrs)
    model_mi = calculate_mi_trapz(t.tensor(mmse_cond), t.tensor(mmse_uncond), t.tensor(logsnrs))
    return {
        'conditional': mmse_cond,
        'unconditional': mmse_uncond,
        'mi': model_mi
    }

theoretical_curves = generate_theoretical_curves(logsnr.cpu().numpy(), X_test, Y_test, num_points=100)
mmse_curves['Theoretical'] = {
        'conditional': theoretical_curves['conditional'],
        'unconditional': theoretical_curves['unconditional'],
        'mi': theoretical_curves['mi'],
        'color': '#50C878'
}